In [1]:
import tensorflow as tf
import numpy as np
import time

# 1. 테스트 설정
DATA_SIZE = 1000  # 데이터 개수
BATCH_SIZE = 32
DELAY = 0.005     # 전처리 1건당 0.005초 지연 (파일 로드나 복잡한 연산 가정)

# 가상 데이터 생성
X = np.random.rand(DATA_SIZE, 100).astype(np.float32)
y = np.random.rand(DATA_SIZE, 1).astype(np.float32)

# 가상의 무거운 전처리 함수
def heavy_preprocess(x, y):
    time.sleep(DELAY) # 의도적 지연
    return x, y

# 방식 1: 일반적인 Python Generator (데이터를 하나씩 순차 처리)
print(f"[방식 1] 일반적인 순차 처리 시작 (지연 시간: {DELAY}s/개)...")
start_time = time.time()

for i in range(0, DATA_SIZE, BATCH_SIZE):
    batch_x, batch_y = [], []
    for j in range(i, min(i + BATCH_SIZE, DATA_SIZE)):
        x_p, y_p = heavy_preprocess(X[j], y[j])
        batch_x.append(x_p)
        batch_y.append(y_p)
    # 여기서 model.train_on_batch(batch_x, batch_y) 등이 일어난다고 가정
    pass 

end_time = time.time()
print(f"==> 일반 방식 소요 시간: {end_time - start_time:.2f}초")

# 방식 2: tf.data 파이프라인 (Parallel Map + Prefetch)
print(f"[방식 2] tf.data 병렬 파이프라인 시작 (AUTOTUNE 적용)...")

# tf.py_function을 사용하여 Python 함수를 텐서플로우 그래프에 포함
def tf_heavy_preprocess(x, y):
    return tf.py_function(heavy_preprocess, [x, y], [tf.float32, tf.float32])

dataset = tf.data.Dataset.from_tensor_slices((X, y))
# num_parallel_calls를 통해 CPU 코어를 여러 개 사용하여 병렬로 전처리
dataset = dataset.map(tf_heavy_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
# 데이터를 묶고, 모델이 학습하는 동안 다음 데이터를 미리 준비
dataset = dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

start_time = time.time()

for batch in dataset:
    # 여기서 모델 학습이 일어난다고 가정
    pass

end_time = time.time()
print(f"==> tf.data 방식 소요 시간: {end_time - start_time:.2f}초")

[방식 1] 일반적인 순차 처리 시작 (지연 시간: 0.005s/개)...
==> 일반 방식 소요 시간: 5.79초
[방식 2] tf.data 병렬 파이프라인 시작 (AUTOTUNE 적용)...
==> tf.data 방식 소요 시간: 0.80초
